<div class="alert alert-info" role="alert"
     style="font-size: 1.4em; padding: 15px; margin: 1px 0; text-align: left; background-color: #d9edf7; border-color: #bce8f1; color: #31708f; border-radius: 1px;">
    Build All_Sargassum_Detections datacube from monthly zarr stores
</div>

### Setup

In [1]:
import numpy as np
import xarray as xr
import pandas as pd

import glob
import os
import re

In [2]:
# Same 0.05-degree TNA grid used by the SGI cubes
lon_1d = np.arange(-70, -20, 0.05)
lat_1d = np.arange(-5,   20, 0.05)

### Find the saved monthly SGI datacubes

In [3]:
zarr_root = '/home/nandii/olci_sg/mo_zarrs'

cube_paths = {}
for path in sorted(glob.glob(os.path.join(zarr_root, 'SGI_*.zarr'))):
    match = re.search(r'SGI_(\d{4})(\d{2})\.zarr', os.path.basename(path))
    year, month = int(match.group(1)), int(match.group(2))
    cube_paths[(year, month)] = path

print(f"{len(cube_paths)} cube(s) found, {min(cube_paths)} to {max(cube_paths)}")

44 cube(s) found, (2016, 5) to (2026, 8)


### Threshold each cube and collapse the `scene` dimension

In [4]:
threshold = -2

# Per-month detection count, scene coverage, and detection frequency on the shared grid
def monthly_counts(zarr_path, threshold):
    ds_cube = xr.open_zarr(zarr_path)
    sgssm_mask = ds_cube['SGI'] >= threshold
    valid_mask = np.isfinite(ds_cube['SGI'])
    detect_count = sgssm_mask.sum(dim = 'scene').values
    cover_count  = valid_mask.sum(dim = 'scene').values
    return detect_count, cover_count

In [5]:
months = sorted(cube_paths)
time_index = pd.to_datetime([f"{y}-{m:02d}-01" for y, m in months])

counts = [monthly_counts(cube_paths[k], threshold) for k in months]
detect_count = np.stack([c[0] for c in counts])
cover_count  = np.stack([c[1] for c in counts])

### Assemble the datacube

In [7]:
detect_frequ = np.where(cover_count > 0, detect_count / cover_count, np.nan)

sargassum_cube = xr.Dataset(
    data_vars = {
        'detect_count' : (['time', 'lat', 'lon'], detect_count),
        'cover_count'  : (['time', 'lat', 'lon'], cover_count ),
        'detect_frequ' : (['time', 'lat', 'lon'], detect_frequ)},
    coords = {'time': time_index, 'lat': lat_1d, 'lon': lon_1d},
    attrs = {
        'title': 'Monthly Sargassum detection datacube from Sentinel-3 OLCI SGI',
        'sgi_threshold' : threshold,
        'source_cubes'  : zarr_root,
        'grid_resolution_deg' : 0.05})

sargassum_cube['detect_count'].attrs['description'] = 'Number of scenes per month/pixel with SGI >= threshold'
sargassum_cube['cover_count'].attrs['description' ] = 'Number of scenes per month/pixel with a finite SGI value'
sargassum_cube['detect_frequ'].attrs['description'] = 'detectcount / cover_count, NaN where cover_count == 0'
sargassum_cube

/tmp/ipykernel_95452/3604906950.py:1: RuntimeWarning: invalid value encountered in divide
  detect_frequ = np.where(cover_count > 0, detect_count / cover_count, np.nan)


<xarray.Dataset> Size: 528MB
Dimensions:       (time: 44, lat: 500, lon: 1000)
Coordinates:
  * time          (time) datetime64[ns] 352B 2016-05-01 ... 2026-08-01
  * lat           (lat) float64 4kB -5.0 -4.95 -4.9 -4.85 ... 19.85 19.9 19.95
  * lon           (lon) float64 8kB -70.0 -69.95 -69.9 ... -20.15 -20.1 -20.05
Data variables:
    detect_count  (time, lat, lon) int64 176MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    cover_count   (time, lat, lon) int64 176MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    detect_frequ  (time, lat, lon) float64 176MB nan nan nan nan ... nan nan nan
Attributes:
    title:                Monthly Sargassum detection datacube from Sentinel-...
    sgi_threshold:        -2
    source_cubes:         /home/nandii/olci_sg/mo_zarrs
    grid_resolution_deg:  0.05

### Save to disk

In [8]:
cube_path = '/home/nandii/olci_sg/All_Sargassum_Detects.zarr'
sargassum_cube.to_zarr(cube_path, mode = 'w')
print(f"Saved to {cube_path}")

Saved to /home/nandii/olci_sg/All_Sargassum_Detects.zarr


/home/nandii/.conda/envs/cloudf/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
